# SIM municipal income indicators 2025 — extraction and recalculation

Extracts SUBDERE's public SIM/BEP workbook and recalculates FCM dependency from its own budget
components. The notebook reads no retired-source baseline; the public workbook is the sole financial
data input.

- Workbook vintage: *Actualizado al 14-06-2026*
- Workbook provenance: `Fuente: CGR`
- Raw workbook: `sample/indicadores-ingresos-municipales-2025.xlsx` (gitignored)


In [ ]:
from pathlib import Path
import openpyxl
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "dataset-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dataset-review").exists(), "run from inside the CityCatalyst-global-data repository"

REVIEWS = ROOT / "dataset-review" / "reviews"
HERE = REVIEWS / "cl-subdere" / "cl-subdere-sim-bep" / "releases" / "2025"
XLSX = HERE / "sample/indicadores-ingresos-municipales-2025.xlsx"
wb = openpyxl.load_workbook(XLSX, data_only=True, read_only=True)
assert "2025" in wb.sheetnames and "Cumplimiento_2025" in wb.sheetnames
banner = str(wb["2025"]["A1"].value)
assert "Fuente:" in banner and "CGR" in banner and "14-06-2026" in banner
print(banner)


## Parse the four-row header and two accounting bases

Data starts at row 5. Money columns appear in cash-received (*percibido*) and accrued (*devengado*)
pairs. The model uses the cash-received basis.


In [ ]:
ws = wb["2025"]
COLS = [
    "comuna_cut", "municipio",
    "ipp_percibido_mclp", "ipp_devengado_mclp",
    "fcm_percibido_mclp", "fcm_devengado_mclp",
    "ip_percibido_mclp", "ip_devengado_mclp",
    "ingresos_totales_percibido_mclp", "ingresos_totales_devengado_mclp",
    "pub_fcm_dependency", "pub_fcm_share_total", "pub_ipp_share_total",
    "pub_ipp_share_total_net_transfers", "fcm_contribution_ratio", "operational_autonomy",
]
raw = pd.DataFrame(
    [row[:16] for row in ws.iter_rows(min_row=5, values_only=True) if row[0] is not None],
    columns=COLS)
raw["comuna_cut"] = raw["comuna_cut"].astype(str).str.strip().str.zfill(5)
raw["municipio"] = raw["municipio"].astype(str).str.strip()
for column in COLS[2:]:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")
print(f"workbook data rows {len(raw)}")


## Remove the non-reporting Antarctic territory and calculate the indicators

The workbook contains one extra all-zero territory row. It is not interpreted as a zero-income comuna.
FCM dependency is recalculated as FCM divided by permanent own income including FCM.


In [ ]:
money = ["ipp_percibido_mclp", "fcm_percibido_mclp", "ingresos_totales_percibido_mclp"]
zero_money = raw[money].fillna(0).sum(axis=1).eq(0)
assert zero_money.sum() == 1, f"expected one all-zero territory row, got {zero_money.sum()}"
df = raw.loc[~zero_money].copy().reset_index(drop=True)

assert (df["ip_percibido_mclp"] -
        (df["ipp_percibido_mclp"] + df["fcm_percibido_mclp"])).abs().max() < 1
df["fcm_dependency"] = df["fcm_percibido_mclp"] / df["ip_percibido_mclp"]
df["fcm_share_total_income"] = (
    df["fcm_percibido_mclp"] / df["ingresos_totales_percibido_mclp"])
df["autonomy"] = (1 - df["fcm_dependency"]).clip(0, 1)

published_delta = (df["fcm_dependency"] - df["pub_fcm_dependency"]).abs()
print(f"published ratio reproduced exactly for {(published_delta <= 1e-9).sum()}/{len(df)} comunas")
print(f"maximum published-column divergence {published_delta.max() * 100:.4f} percentage points")


## Validate the public-source output

The capacity-tier CUT list is used only as a commercially reusable project-universe check; none of its
values are joined to this output.


In [ ]:
assert len(df) == 345, f"expected 345 comunas, got {len(df)}"
assert df["comuna_cut"].is_unique, "duplicate CUT"
assert df["comuna_cut"].str.fullmatch(r"\d{5}").all(), "invalid CUT"
assert df[["fcm_dependency", "autonomy"]].notna().all().all()
assert df["fcm_dependency"].between(0, 1).all()
assert df["autonomy"].between(0, 1).all()
assert (df["fcm_dependency"] >= df["fcm_share_total_income"] - 1e-9).all()

capacity_keys = pd.read_csv(
    REVIEWS / "oef/cl-municipal-capacity-tier/releases/v1/data/municipal_capacity_tier.csv",
    dtype={"comuna_cut": str}, usecols=["comuna_cut"])["comuna_cut"].str.zfill(5)
assert set(df["comuna_cut"]) == set(capacity_keys), "CUT universe differs from capacity release"
print("structural assertions passed — exact 345-CUT project universe")


## Export


In [ ]:
OUT_COLS = [
    "comuna_cut", "municipio",
    "ipp_percibido_mclp", "fcm_percibido_mclp", "ip_percibido_mclp",
    "ingresos_totales_percibido_mclp", "ipp_devengado_mclp", "fcm_devengado_mclp",
    "ip_devengado_mclp", "ingresos_totales_devengado_mclp", "fcm_dependency",
    "fcm_share_total_income", "autonomy", "fcm_contribution_ratio", "operational_autonomy",
]
out = df[OUT_COLS].copy()
for column in ["fcm_dependency", "fcm_share_total_income", "autonomy"]:
    out[column] = out[column].round(6)
out["source_vintage"] = "2026-06-14"
out = out.sort_values("comuna_cut").reset_index(drop=True)
out.to_csv(HERE / "data/sim_municipal_income_2025.csv", index=False)
print(f"wrote data/sim_municipal_income_2025.csv — {out.shape[0]} rows x {out.shape[1]} columns")


## Result

The cleaned output contains a reported autonomy value for every one of the 345 project comunas, with
no median fill and no retired-source data. Promotion caveats are recorded in `review.md`.
